Feature Extraction
Goal: Convert preprocessed EEG trials into numerical features. 

Feature represenations in use: Common Spatial Patterns (CSP), Band Power, Hjorth Parameter

In [17]:
import mne
import numpy as np

from mne.decoding import CSP

file_path = "/Users/vaneeza/EEG_BCI_Project/data/BCICIV_2a_gdf/A01T.gdf"
raw = mne.io.read_raw_gdf(file_path, preload=True)
print (raw)

# event identification
events, event_id = mne.events_from_annotations(raw, event_id = "auto")
print (event_id)

left_hand_code = event_id["769"]
right_hand_code = event_id["770"]
print("Left Hand Code:", left_hand_code)
print("Right Hand Code:", right_hand_code)

motor_events = {
    "left_hand": left_hand_code,
    "right_hand": right_hand_code
}

epochs = mne.Epochs(
    raw, events, event_id=motor_events, 
    tmin = -0.5, tmax =4.0, baseline = (-0.5,0), preload = True)
epochs.filter( l_freq = 8, h_freq = 30)
epochs.crop(tmin = 0.5, tmax = 3.5)
eog_channels = [
    "EOG-left",
    "EOG-central",
    "EOG-right"
]

epochs.drop_channels(eog_channels)

print("Number of EEG channels:", len(epochs.ch_names))

X = epochs.get_data()
y = epochs.events[:,2]
print("EEG shape:", X.shape)

print("Labels shape:", y.shape)

csp = CSP(
    n_components=6,
    log=True
)
# CSP will create 6 new features from the original EEG channels to separate the classes

x_csp = csp.fit_transform(X, y)
print("CSP feature shape:", x_csp.shape)

# Band Power = how much signal power exists in a specific frequency range
# Mu Band (8-13 Hz) and Beta Band (13-30 Hz)

print("X shape:", X.shape)
print("Number of trials in X:", len(X))
print("Number of channels in first trial:", len(X[0]))

sfreq = raw.info["sfreq"]

bandpower_features = []

for trial in X:

    trial_features = []

    for channel in trial:

        psd, freqs = mne.time_frequency.psd_array_welch(
            channel,
            sfreq=sfreq,
            fmin=8,
            fmax=30,
            verbose=False
        )

        mu_mask = (freqs >= 8) & (freqs < 13)
        beta_mask = (freqs >= 13) & (freqs <= 30)

        mu_power = psd[mu_mask].mean()
        beta_power = psd[beta_mask].mean()

        trial_features.append(mu_power)
        trial_features.append(beta_power)

    bandpower_features.append(trial_features)

x_bandpower = np.array(bandpower_features)

print("Band Power feature shape:", x_bandpower.shape)

Extracting GDF parameters from /Users/vaneeza/EEG_BCI_Project/data/BCICIV_2a_gdf/A01T.gdf...
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
EEG-Fz, EEG, EEG, EEG, EEG, EEG, EEG, EEG-C3, EEG, EEG-Cz, EEG, EEG-C4, EEG, EEG, EEG, EEG, EEG, EEG, EEG, EEG-Pz, EEG, EEG, EOG-left, EOG-central, EOG-right
Creating raw.info structure...
Reading 0 ... 672527  =      0.000 ...  2690.108 secs...


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


<RawGDF | A01T.gdf, 25 x 672528 (2690.1 s), ~128.3 MiB, data loaded>
Used Annotations descriptions: [np.str_('1023'), np.str_('1072'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('769'), np.str_('770'), np.str_('771'), np.str_('772')]
{np.str_('1023'): 1, np.str_('1072'): 2, np.str_('276'): 3, np.str_('277'): 4, np.str_('32766'): 5, np.str_('768'): 6, np.str_('769'): 7, np.str_('770'): 8, np.str_('771'): 9, np.str_('772'): 10}
Left Hand Code: 7
Right Hand Code: 8
Not setting metadata
144 matching events found
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 144 events and 1126 original time points ...
0 bad epochs dropped
Setting up band-pass filter from 8 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower pass

In [18]:
# Hjorth Parameters
def calculate_hjorth(signal):

    activity = np.var(signal)

    first_derivative = np.diff(signal)

    mobility = np.sqrt(
        np.var(first_derivative) / activity
    )

    second_derivative = np.diff(first_derivative)

    mobility_derivative = np.sqrt(
        np.var(second_derivative) /
        np.var(first_derivative)
    )

    complexity = mobility_derivative / mobility

    return activity, mobility, complexity

hjorth_features = []

for trial in X:

    trial_features = []

    for channel in trial:

        activity, mobility, complexity = calculate_hjorth(channel)

        trial_features.append(activity)
        trial_features.append(mobility)
        trial_features.append(complexity)

    hjorth_features.append(trial_features)

x_hjorth = np.array(hjorth_features)

print("Hjorth feature shape:", x_hjorth.shape)



Hjorth feature shape: (144, 66)


In [19]:
print("Original EEG shape:", X.shape)
print("CSP shape:", x_csp.shape)
print("Band Power shape:", x_bandpower.shape)
print("Hjorth shape:", x_hjorth.shape)

Original EEG shape: (144, 22, 751)
CSP shape: (144, 6)
Band Power shape: (144, 44)
Hjorth shape: (144, 66)


Comparing Feature Representations
Original EEG: Full Voltage Signals over time
CSP: Reduce each trial to a small number of spatial features
Band Power: summarizes mu and beta activity
Hjorth Parameters: summarize time-domain porperties of each channel